# Meridian monthly report

Completed orders and revenue for one month, by country and by category.

In [ ]:
# Parameters: papermill replaces these values
month = '2023-09'

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

URL = ('postgresql+psycopg2://meridian_user:meridian2024'
       '@localhost:5432/meridian_db')
engine = create_engine(URL)
orders = pd.read_sql("""
    SELECT o.order_id, o.status, o.total_amount::float AS amount,
           co.country_name, ca.category_name
    FROM fact_orders o
    JOIN dim_customer c  USING (customer_id)
    JOIN dim_country co  USING (country_id)
    JOIN dim_product p   USING (product_id)
    JOIN dim_category ca USING (category_id)
    WHERE to_char(o.order_date, 'YYYY-MM') = %(month)s
""", engine, params={'month': month})
len(orders)

## Headline figures

In [ ]:
done = orders[orders['status'] == 'completed']
print(f'{month}: {len(done):,} completed orders, revenue £{done.amount.sum():,.2f}')
print(f'Return rate: {(orders.status == "returned").mean():.1%}')

## Revenue by country

In [ ]:
by_country = done.groupby('country_name')['amount'].sum().sort_values(ascending=False)
ax = by_country.div(1e3).plot.barh(color='#14213D', figsize=(6, 3), title=f'Revenue by country, {month} (£ thousand)')
ax.invert_yaxis()
by_country.round(2)